# Chapter 5 — Tools as Typed Actions

*A tool is not just a function. It is a registered, schema-validated, risk-classified capability.*

## Objective

Register three mock tools with explicit input and output schemas. Try three router families on the same query set. Inspect which router routes correctly.

In [ ]:
from pydantic import BaseModel
from glassloop.tools import (
    EmbeddingRouter, LMRouter, RiskLevel, Router, RuleRouter,
    Tool, ToolRegistry,
)

## Define typed tools

Each tool has a name, description, input schema, output schema and risk level. Without schemas, a model can invent argument names and the system has no way to reject them cleanly.

In [ ]:
class SearchIn(BaseModel):
    query: str
class SearchOut(BaseModel):
    results: list[dict]

class CalcIn(BaseModel):
    expression: str
class CalcOut(BaseModel):
    result: float

class EmailIn(BaseModel):
    to: str
    subject: str
    body: str
class EmailOut(BaseModel):
    sent: bool

search_tool = Tool(name='search', description='search documents',
                   input_schema=SearchIn, output_schema=SearchOut, risk=RiskLevel.LOW)
calc_tool = Tool(name='calculator', description='evaluate basic arithmetic',
                 input_schema=CalcIn, output_schema=CalcOut, risk=RiskLevel.LOW)
email_tool = Tool(name='send_email', description='send an email to a recipient',
                  input_schema=EmailIn, output_schema=EmailOut, risk=RiskLevel.HIGH)

## Register them and try validation

In [ ]:
registry = ToolRegistry()
registry.register(search_tool)
registry.register(calc_tool)
registry.register(email_tool)
print('registered:', [t.name for t in registry.all()])

try:
    registry.validate('calculator', {'expression': 'not-an-expression', 'extra': 1})
    print('still ok (extra ignored)')
except Exception as e:
    print('validation failed:', e)

try:
    registry.validate('calculator', {'wrong': 'no expression field'})
except Exception as e:
    print('missing field rejected')

## Three routers, one benchmark

RuleRouter is transparent but brittle. EmbeddingRouter generalizes but needs calibration. LMRouter is flexible but adds latency. We compare them on a small benchmark.

In [ ]:
import hashlib

class MockEmbedder:
    dim = 16
    def embed(self, texts):
        out = []
        for t in texts:
            h = hashlib.sha256(t.lower().encode()).digest()
            out.append([b / 255.0 for b in h[:self.dim]])
        return out

class FixedLM:
    def __init__(self, mapping):
        self._mapping = mapping
    def complete(self, prompt, **kw):
        for key, name in self._mapping.items():
            if key in prompt.lower():
                return name
        return 'NONE'
    def token_count(self, t):
        return len(t.split())

rule = RuleRouter({'search': 'search', 'calculate': 'calculator', 'compute': 'calculator', 'email': 'send_email'})
emb = EmbeddingRouter(MockEmbedder(), threshold=0.0)
lm = LMRouter(FixedLM({'find a document': 'search', 'compute': 'calculator', 'compose a message': 'send_email'}))

In [ ]:
benchmark = [
    ('please search the policy docs', 'search'),
    ('compute 2 plus 2', 'calculator'),
    ('email alice the report', 'send_email'),
    ('find a document about overdrafts', 'search'),
    ('chat about the weather', None),
]

def benchmark_router(name, router):
    correct = 0
    for query, expected in benchmark:
        picked = router.route(query, registry)
        picked_name = picked.name if picked else None
        marker = 'OK' if picked_name == expected else 'XX'
        print(f'  [{marker}] {query!r:<45} -> {picked_name!r}  expected={expected!r}')
        if picked_name == expected: correct += 1
    print(f'  {name}: {correct}/{len(benchmark)}')

for n, r in [('RuleRouter', rule), ('EmbeddingRouter', emb), ('LMRouter', lm)]:
    print(n)
    benchmark_router(n, r)

## When to pick each router

- **Rule-based** when the keyword space is small and well-known. Transparent in audit.
- **Embedding-based** when the user phrasing varies. Needs threshold calibration.
- **LLM-based** when tool selection requires reasoning about the user's intent. Adds latency and cost.
- **Hybrid** in practice: rules for the easy cases, embeddings or LLM as fallback.

## Anti-patterns flagged here

- Tools defined as raw functions without schemas.
- Letting the model invent argument names.
- Regexing the model output to find the tool call.

In [ ]:
# Self-check
assert isinstance(rule, Router) and isinstance(emb, Router) and isinstance(lm, Router)
assert rule.route('please search the docs', registry).name == 'search'
print('OK')